In [1]:

from pathlib import Path
import duckdb
import polars as pl

# Folder utama proyek
PROJECT_ROOT = Path.cwd().resolve().parent

# Lokasi dataset mentah
DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Lokasi penyimpanan dataset bersih
OUTPUT_PATH = (
    PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
)

# Cari semua file Parquet
files = sorted(DATA_PATH.glob("*.parquet"))

if not files:
    raise FileNotFoundError(
        f"Tidak ada file Parquet di {DATA_PATH}"
    )

# Baca dan gabungkan kedua file
df = pl.read_parquet([str(file) for file in files])

print(f"Jumlah file: {len(files)}")
print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print("\nNama dan tipe kolom:")
print(df.schema)

display(df.head())


Jumlah file: 2
Ukuran awal: 5,566,778 baris x 1 kolom

Nama dan tipe kolom:
Schema({'text': String})


text
str
"""Indonesia adalah negara kepula…"
"""Secara geografis, Indonesia be…"
"""Indonesia memiliki iklim tropi…"
"""Sejarah Indonesia sangat panja…"
"""Indonesia memproklamasikan kem…"


In [2]:

connection = duckdb.connect()

# Daftarkan DataFrame Polars ke DuckDB
connection.register("dataset", df)

# Ringkasan statistik seluruh kolom
profil = connection.execute(
    "SUMMARIZE SELECT * FROM dataset"
).pl()

display(profil)


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""text""","""VARCHAR""","""""","""􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁…",4512985,null,null,null,null,null,5566778,0.00


In [3]:

jumlah_baris = len(df)

null_summary = (
    df.null_count()
    .transpose(
        include_header=True,
        header_name="kolom",
        column_names=["jumlah_null"]
    )
    .with_columns(
        (pl.col("jumlah_null") / jumlah_baris * 100)
        .round(2)
        .alias("persen_null")
    )
    .filter(pl.col("jumlah_null") > 0)
    .sort("jumlah_null", descending=True)
)

display(null_summary)

print("Total nilai null:", df.null_count().sum_horizontal().sum())


kolom,jumlah_null,persen_null
str,u32,f64


Total nilai null: 0


In [4]:

print("Jumlah baris:", df.height)
print("Jumlah kolom:", df.width)
print("\nTipe data setiap kolom:")
print(df.schema)

print("\nJumlah nilai unik setiap kolom:")
print(df.n_unique())

print("\nJumlah baris duplikat seluruh kolom:")
print(df.height - df.unique().height)


Jumlah baris: 5566778
Jumlah kolom: 1

Tipe data setiap kolom:
Schema({'text': String})

Jumlah nilai unik setiap kolom:
5281769

Jumlah baris duplikat seluruh kolom:
285009


### Catatan Milestone 2: Data Quality dan Profiling

Dataset yang digunakan terdiri dari dua file Parquet dengan total 5.566.778 baris dan satu kolom bernama `text` bertipe String. Dataset berisi teks berbahasa Indonesia dan sesuai untuk analisis korpus teks.

Hasil pemeriksaan menunjukkan tidak ditemukan nilai null, sehingga belum diperlukan penanganan missing values. Namun, ditemukan 285.009 baris duplikat berdasarkan kesamaan seluruh isi kolom, atau sekitar 5,12% dari total baris. Duplikasi tersebut perlu dievaluasi sebelum dilakukan penghapusan.

Karena dataset hanya memiliki kolom teks, metode outlier berbasis IQR tidak dapat diterapkan langsung pada kolom tersebut. Pembersihan selanjutnya dapat difokuskan pada pemeriksaan teks kosong, duplikasi, serta validasi kualitas teks. Setelah proses pembersihan selesai, jumlah baris sebelum dan sesudah cleaning perlu dibandingkan, lalu hasilnya disimpan dalam format Parquet untuk digunakan pada Milestone 3.
